# Natural Resources Pipeline — Production Values Version

Builds `natural_resources_production_values.csv` from six sources:

| Source | Label | Content |
|---|---|---|
| EI Narrow CSV | `EI_CSV` | Oil, Gas, Coal production/consumption; Cobalt, Lithium, Graphite, Rare Earth production/reserves |
| EI Excel workbook (`base_dataset.xlsx`) | `EI_Excel` | Mineral P-R sheets (13 minerals) |
| EI Excel price sheets | `EI_Prices` | Oil (1861+), Coal (NW Europe), Cobalt/Lithium/Nickel/Graphite prices |
| OWID Mineral CSVs | `OWID` | Production, reserves, unit-value prices for 20+ minerals |
| USGS ds140 files | `USGS` | Mineral unit values ($/t) — graceful no-op if `rawdata/USGS/` absent |
| Oil/Gas/Coal benchmarks | `GasPrice` | Brent ($/bbl), Coal PCOALAUUSDA ($/t), and Henry Hub/TTF/JKM gas ($/MMBtu) averaged into per-resource world price series |
| Consolidated prices | `ConsolidatedPrices` | Pre-harmonised wide-format file covering all 19 minerals + Oil + Coal + Gas, 1990–2024 (`workingdata/natural resource prices.xlsx`) |

**Source priority** (highest → lowest):
- All prices: `ConsolidatedPrices` → `USGS` → `EI_Prices` → `GasPrice` → `OWID` → `EI_Excel` → `EI_CSV`
- Production/consumption/reserves: per-country coverage score (years of non-null data) governs; global priority breaks ties only

**Conflict resolution:**
For price rows (world-level), source priority determines which value is kept.
For production/reserves rows, the source with the most years of data for that specific country wins, regardless of global ranking. This allows OWID to beat EI for countries where OWID has better longitudinal coverage and vice versa.

**Known remaining gaps:**
- Coal Reserves, Oil Reserves, Natural Gas Reserves (no source data)
- `rawdata/USGS/` absent on disk — add `ds140-*.xlsx` files to activate USGS mineral prices

In [1]:
import sys, os
if "scripts" not in sys.path: sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import re

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)
from standardize_country import add_iso3, ALIAS_TO_ISO3, ISO3_TO_WB

## Paths

In [2]:
# In Jupyter the kernel's cwd can differ from the notebook's directory.
# We resolve to absolute and verify that rawdata/ exists; if not, fall back
# to the standard Capstone location so paths work regardless of where the
# kernel was launched.
BASE = Path('.').resolve()
if not (BASE / 'rawdata').is_dir():
    _fallback = Path.home() / 'Desktop' / 'GitHub' / 'Capstone'
    if (_fallback / 'rawdata').is_dir():
        BASE = _fallback
        print(f"NOTE: Working directory resolved to {BASE}")
    else:
        raise FileNotFoundError(
            f"Cannot find rawdata/ in {Path('.').resolve()} or {_fallback}. "
            "Set BASE manually in this cell."
        )

RAW      = BASE / 'rawdata'
INTER    = BASE / 'intermediary'

MINERALS_DIR = RAW / 'Minerals'
EI_CSV_PATH  = RAW / 'Statistical Review of World Energy Narrow File-1.csv'
EI_XLSX_PATH = RAW / 'base_dataset.xlsx'

# New price sources
USGS_DIR       = RAW / 'USGS'                       # folder with ds140-*.xlsx files
GAS_PRICE_PATH = RAW / 'Oil Gas Coal Uranium Price.xlsx'

# Consolidated price file (minerals + oil/coal/gas benchmarks in one wide-format sheet).
# Located in workingdata/ rather than rawdata/ as it is a compiled working file.
# If running from a subdirectory (e.g. FINAL CODE RECAP/), workingdata/ lives one level up.
# NOTE: filename uses spaces, not underscores — 'natural resource prices.xlsx'
WORK = BASE / 'workingdata'
if not WORK.is_dir():
    WORK = BASE.parent / 'workingdata'
CONSOL_PRICE_PATH = WORK / 'natural resource prices.xlsx'

# Outputs
OUTPUT_PATH       = INTER / 'natural_resources_production_values.csv'
PRODVAL_PATH      = INTER / 'natural_resources_production_values.csv'

INTER.mkdir(parents=True, exist_ok=True)

for label, p in [
    ('EI CSV',             EI_CSV_PATH),
    ('EI Excel',           EI_XLSX_PATH),
    ('Minerals dir',       MINERALS_DIR),
    ('Gas price file',     GAS_PRICE_PATH),
    ('Consolidated prices',CONSOL_PRICE_PATH),
]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f"  {status}  {label}: {p}")
print(f"  {'✓' if USGS_DIR.exists() else '✗ MISSING (graceful no-op)'}  USGS dir: {USGS_DIR}")

  ✓  EI CSV: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis/rawdata/Statistical Review of World Energy Narrow File-1.csv
  ✓  EI Excel: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis/rawdata/base_dataset.xlsx
  ✓  Minerals dir: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis/rawdata/Minerals
  ✓  Gas price file: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis/rawdata/Oil Gas Coal Uranium Price.xlsx
  ✓  Consolidated prices: /Users/leoss/Desktop/GitHub/Capstone/Client/workingdata/natural resource prices.xlsx
  ✓  USGS dir: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis/rawdata/USGS


## Mappings

In [3]:
# EI Narrow CSV variable codes -> (Resource, Metric)
EI_CSV_VAR_MAP = {
    'oilprod_kbd':      ('Oil',             'Production'),
    'oilcons_kbd':      ('Oil',             'Consumption'),
    'gasprod_bcm':      ('Natural Gas',     'Production'),
    'gascons_bcm':      ('Natural Gas',     'Consumption'),
    'coalprod_mt':      ('Coal',            'Production'),
    'coalcons_ej':      ('Coal',            'Consumption'),
    'cobalt_kt':        ('Cobalt',          'Production'),
    'cobaltres_kt':     ('Cobalt',          'Reserves'),
    'lithium_kt':       ('Lithium',         'Production'),
    'lithiumres_kt':    ('Lithium',         'Reserves'),
    'graphite_kt':      ('Natural Graphite','Production'),
    'graphiteres_kt':   ('Natural Graphite','Reserves'),
    'rareearths_kt':    ('Rare Earth',      'Production'),
    'rareearthsres_kt': ('Rare Earth',      'Reserves'),
}

EXCEL_PR_SHEETS = {
    'Cobalt P-R':              'Cobalt',
    'Lithium P-R':             'Lithium',
    'Natural Graphite P-R':    'Natural Graphite',
    'Rare Earth metals P-R':   'Rare Earth',
    'Copper P-R':              'Copper',
    'Manganese P-R':           'Manganese',
    'Nickel P-R':              'Nickel',
    'Zinc P-R':                'Zinc',
    'Platinum Group Metals P-R': 'Platinum Group',
    'Bauxite P-R':             'Bauxite',
    'Aluminium P-R':           'Aluminium',
    'Tin P-R':                 'Tin',
    'Vanadium P-R':            'Vanadium',
}

OWID_FILE_MAP = {
    'aluminum-production':                        ('Aluminium',           'Production'),
    'aluminum-unit-value':                        ('Aluminium',           'Price'),
    'bauxite-production':                         ('Bauxite',             'Production'),
    'bauxite-unit-value':                         ('Bauxite',             'Price'),
    'coal-production':                            ('Coal',                'Production'),
    'cobalt-production':                          ('Cobalt',              'Production'),
    'cobalt-reserves':                            ('Cobalt',              'Reserves'),
    'cobalt-unit-value':                          ('Cobalt',              'Price'),
    'copper-production':                          ('Copper',              'Production'),
    'copper-reserves':                            ('Copper',              'Reserves'),
    'copper-unit-value':                          ('Copper',              'Price'),
    'gold-production':                            ('Gold',                'Production'),
    'gold-reserves':                              ('Gold',                'Reserves'),
    'gold-reserves.filtered':                     ('Gold',                'Reserves'),
    'gold-unit-value':                            ('Gold',                'Price'),
    'graphite-production':                        ('Natural Graphite',    'Production'),
    'graphite-reserves':                          ('Natural Graphite',    'Reserves'),
    'graphite-unit-value':                        ('Natural Graphite',    'Price'),
    'iron-ore-crude-ore-production':              ('Iron ore',            'Production'),
    'iron-ore-crude-ore-unit-value':              ('Iron ore',            'Price'),
    'lead-production':                            ('Lead',                'Production'),
    'lead-reserves':                              ('Lead',                'Reserves'),
    'lead-unit-value':                            ('Lead',                'Price'),
    'lithium-production':                         ('Lithium',             'Production'),
    'lithium-reserves':                           ('Lithium',             'Reserves'),
    'lithium-unit-value':                         ('Lithium',             'Price'),
    'magnesium-compounds-production':             ('Magnesium compounds', 'Production'),
    'magnesium-compounds-unit-value':             ('Magnesium compounds', 'Price'),
    'manganese-production':                       ('Manganese',           'Production'),
    'manganese-reserves':                         ('Manganese',           'Reserves'),
    'manganese-unit-value':                       ('Manganese',           'Price'),
    'nickel-production':                          ('Nickel',              'Production'),
    'nickel-reserves':                            ('Nickel',              'Reserves'),
    'nickel-unit-value':                          ('Nickel',              'Price'),
    'petroleum-production':                       ('Oil',                 'Production'),
    'platinum-group-metals-palladium-production': ('Platinum Group',      'Production'),
    'platinum-group-metals-palladium-reserves':   ('Platinum Group',      'Reserves'),
    'rare-earths-production':                     ('Rare Earth',          'Production'),
    'rare-earths-reserves':                       ('Rare Earth',          'Reserves'),
    'rare-earths-unit-value':                     ('Rare Earth',          'Price'),
    'refined-cadmium-production':                 ('Cadmium',             'Production'),
    'refined-cadmium-production-1':               ('Cadmium',             'Production'),
    'refined-cadmium-unit-value':                 ('Cadmium',             'Price'),
    'silver-production':                          ('Silver',              'Production'),
    'silver-reserves':                            ('Silver',              'Reserves'),
    'silver-reserves-1':                          ('Silver',              'Reserves'),
    'silver-unit-value':                          ('Silver',              'Price'),
    'tin-production':                             ('Tin',                 'Production'),
    'tin-reserves':                               ('Tin',                 'Reserves'),
    'tin-unit-value':                             ('Tin',                 'Price'),
    'vanadium-production':                        ('Vanadium',            'Production'),
    'vanadium-reserves':                          ('Vanadium',            'Reserves'),
    'vanadium-unit-value':                        ('Vanadium',            'Price'),
    'zinc-production':                            ('Zinc',                'Production'),
    'zinc-reserves':                              ('Zinc',                'Reserves'),
    'zinc-unit-value.filtered':                   ('Zinc',                'Price'),
}

OWID_METRIC_PATTERNS = {
    'production':  'Production',
    'unit value':  'Price',
    'reserves':    'Reserves',
    'consumption': 'Consumption',
}

FINAL_COLS = ['Country', 'Year', 'Value', 'Resource', 'Metric']

# ds140 filenames use short mineral codes; map to pipeline resource names.
# Extend this as needed when new ds140 files appear in USGS_DIR.
USGS_NAME_MAP = {
    'alumi': 'Aluminium',
    'bauxi': 'Bauxite',
    'cadmi': 'Cadmium',
    'cobal': 'Cobalt',
    'coppe': 'Copper',
    'gold':  'Gold',
    'graph': 'Natural Graphite',
    'iron':  'Iron ore',
    'lead':  'Lead',
    'lithi': 'Lithium',
    'magne': 'Magnesium compounds',
    'manga': 'Manganese',
    'nicke': 'Nickel',
    'plati': 'Platinum Group',
    'rare':  'Rare Earth',
    'silve': 'Silver',
    'tin':   'Tin',
    'vanad': 'Vanadium',
    'zinc':  'Zinc',
}

## Unit conversions

In [4]:
# OWID reports production in tonnes. Convert to EI canonical units.
# Oil: tonnes/yr -> kbd. Coal: tonnes/yr -> Mt. Minerals: tonnes -> kt.
# Price rows excluded (already in $/unit).
OWID_TO_CANONICAL = {
    'Oil':                  1 / (365 * 1_000 * 0.136),
    'Coal':                 1 / 1_000_000,
    'Aluminium':            1 / 1_000,
    'Bauxite':              1 / 1_000,
    'Cadmium':              1 / 1_000,
    'Cobalt':               1 / 1_000,
    'Copper':               1 / 1_000,
    'Gold':                 1 / 1_000,
    'Iron ore':             1 / 1_000,
    'Lead':                 1 / 1_000,
    'Lithium':              1 / 1_000,
    'Magnesium compounds':  1 / 1_000,
    'Manganese':            1 / 1_000,
    'Natural Graphite':     1 / 1_000,
    'Nickel':               1 / 1_000,
    'Platinum Group':       1 / 1_000,
    'Rare Earth':           1 / 1_000,
    'Silver':               1 / 1_000,
    'Tin':                  1 / 1_000,
    'Vanadium':             1 / 1_000,
    'Zinc':                 1 / 1_000,
}

OWID_RESERVES_ALREADY_KT = set()

# Maps (production_unit, price_unit) -> multiplier so that
#   ProductionValue_USD = Production * Price * multiplier
# Oil: kbd * 1000 * 365 gives annual barrels.
# Gas: bcm * 35.315e6 * 1.037 gives MMBtu (1 bcm = 35.315bn cf; 1 Mcf ~ 1.037 MMBtu).
# Coal/minerals: Mt * 1e6 or kt * 1e3 to get tonnes.

PRODVAL_MULTIPLIER = {
    'Oil':          365_000,            # kbd -> bbl/yr
    'Natural Gas':  35_315_000 * 1.037, # bcm -> MMBtu
    'Coal':         1_000_000,          # Mt -> tonnes
}
# All minerals default to 1_000 (kt -> tonnes)
MINERAL_PRODVAL_MULT = 1_000

## Source priority

In [5]:
# Source priority order for each (resource, metric) combination.
# ConsolidatedPrices is #1 for all prices (widest coverage, pre-harmonised).
# GasPrice now covers Natural Gas, Oil (Brent), and Coal — ranked below EI_Prices
# so it only fills year gaps not already in the EI series.

_EI_HYDRO          = ['EI_CSV', 'OWID', 'EI_Excel', 'EI_Prices']
_EI_HYDRO_RESERVES = ['OWID', 'EI_CSV', 'EI_Excel', 'EI_Prices']
_MINERAL_PROD      = ['OWID', 'EI_Excel', 'EI_CSV', 'EI_Prices']
_MINERAL_RES       = ['OWID', 'EI_Excel', 'EI_CSV', 'EI_Prices']
_PRICE             = ['ConsolidatedPrices', 'USGS', 'EI_Prices', 'OWID', 'EI_Excel', 'EI_CSV']

RESOURCE_PRIORITY = {
    ('Oil',          'Production'):  _EI_HYDRO,
    ('Oil',          'Consumption'): _EI_HYDRO,
    ('Oil',          'Reserves'):    _EI_HYDRO_RESERVES,
    ('Oil',          'Price'):       ['ConsolidatedPrices', 'EI_Prices', 'GasPrice', 'OWID', 'EI_Excel', 'EI_CSV'],
    ('Natural Gas',  'Production'):  _EI_HYDRO,
    ('Natural Gas',  'Consumption'): _EI_HYDRO,
    ('Natural Gas',  'Reserves'):    _EI_HYDRO_RESERVES,
    ('Natural Gas',  'Price'):       ['ConsolidatedPrices', 'GasPrice', 'EI_Prices', 'OWID', 'EI_Excel', 'EI_CSV'],
    ('Coal',         'Production'):  _EI_HYDRO,
    ('Coal',         'Consumption'): _EI_HYDRO,
    ('Coal',         'Reserves'):    _EI_HYDRO_RESERVES,
    ('Coal',         'Price'):       ['ConsolidatedPrices', 'EI_Prices', 'GasPrice', 'USGS', 'OWID', 'EI_Excel', 'EI_CSV'],
    ('Platinum Group', 'Production'): ['EI_Excel', 'OWID', 'EI_CSV', 'EI_Prices'],
}

def _priority(resource: str, metric: str, source: str) -> int:
    order = RESOURCE_PRIORITY.get((resource, metric))
    if order is None:
        order = _PRICE if metric == 'Price' else (_MINERAL_RES if metric == 'Reserves' else _MINERAL_PROD)
    try:
        return order.index(source)
    except ValueError:
        return len(order)

## Source 1 -- EI Narrow CSV

In [6]:
def load_ei_csv(path: Path) -> pd.DataFrame:
    logger.info('Loading EI Narrow CSV...')
    if not path.exists():
        logger.warning(f'EI CSV not found: {path}')
        return pd.DataFrame(columns=FINAL_COLS)

    df = pd.read_csv(path)
    frames = []
    for var_code, (resource, metric) in EI_CSV_VAR_MAP.items():
        subset = df[df['Var'] == var_code][['Country', 'Year', 'Value']].copy()
        if subset.empty:
            logger.warning(f'  Var code not found: {var_code}')
            continue
        subset['Resource'] = resource
        subset['Metric']   = metric
        frames.append(subset)

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    result = pd.concat(frames, ignore_index=True)
    logger.info(f'  EI CSV: {len(result):,} rows across {len(frames)} variables')
    return result[FINAL_COLS]

## Source 2a -- EI Excel mineral P-R sheets

In [7]:
DECADE_RE = re.compile(r'^\d{4}-\d{2,4}$')

skip_patterns = re.compile(
    r'(total|source|rest of|^\u2666|growth|share|r/p|billion|million|thousand|'
    r'^nan$|^\s*$|production|reserves|mine|content|'
    r'change|delta|percent|increase|decrease|variation|footnote|^\d|^of which)',
    re.IGNORECASE,
)


def _parse_pr_sheet(xl: pd.ExcelFile, sheet: str, resource: str) -> pd.DataFrame:
    df   = pd.read_excel(xl, sheet_name=sheet, header=None)
    row1 = df.iloc[1]
    row2 = df.iloc[2]

    decade_col = None
    for ci, val in enumerate(row2):
        if DECADE_RE.match(str(val).strip()):
            decade_col = ci
            break
    excluded = {decade_col - 1, decade_col, decade_col + 1} if decade_col is not None else set()

    year_cols = {}
    for ci, val in enumerate(row2):
        if ci in excluded:
            continue
        try:
            yr = int(float(val))
            if 1900 < yr < 2100:
                year_cols[ci] = yr
        except (ValueError, TypeError):
            pass

    reserves_col  = None
    reserves_year = None
    for ci, val in enumerate(row1):
        if isinstance(val, str) and re.search(r'reserves|capacity', val, re.IGNORECASE):
            reserves_col  = ci
            m             = re.search(r'(\d{4})', str(row2.iloc[ci]))
            reserves_year = int(m.group(1)) if m else None
            break

    data_rows = []
    for i, row in df.iterrows():
        if i <= 3:
            continue
        val = str(row.iloc[0]).strip()
        if skip_patterns.search(val):
            continue
        data_rows.append(i)

    frames = []

    if year_cols and data_rows:
        prod_df = df.iloc[data_rows, [0] + list(year_cols.keys())].copy()
        prod_df.columns = ['Country'] + [year_cols[c] for c in year_cols.keys()]
        prod_df = prod_df.melt(id_vars='Country', var_name='Year', value_name='Value')
        prod_df['Resource'] = resource
        prod_df['Metric']   = 'Production'
        frames.append(prod_df)

    if reserves_col is not None and reserves_year is not None and data_rows:
        res_df = df.iloc[data_rows, [0, reserves_col]].copy()
        res_df.columns = ['Country', 'Value']
        res_df['Year']     = reserves_year
        res_df['Resource'] = resource
        res_df['Metric']   = 'Reserves'
        frames.append(res_df)

    if not frames:
        logger.warning(f'  No data parsed from sheet: {sheet}')
        return pd.DataFrame(columns=FINAL_COLS)

    out = pd.concat(frames, ignore_index=True)
    out['Value'] = pd.to_numeric(out['Value'], errors='coerce')

    n_neg = (out['Value'] < 0).sum()
    if n_neg > 0:
        logger.warning(
            f'  {sheet}: {n_neg} unexpected negative(s) after growth-column exclusion '
            f'-- inspect sheet layout.'
        )
        out = out[out['Value'] >= 0]

    return out[FINAL_COLS]

In [8]:
def load_ei_excel_minerals(path: Path) -> pd.DataFrame:
    logger.info('Loading EI Excel mineral P-R sheets...')
    if not path.exists():
        logger.warning(f'EI Excel not found: {path}')
        return pd.DataFrame(columns=FINAL_COLS)

    xl = pd.ExcelFile(path)

    EXCEL_METRIC_SCALE = {
        ('Platinum Group', 'Production'): 1 / 1_000,
        ('Platinum Group', 'Reserves'):   1 / 1_000,
    }

    frames = []
    for sheet, resource in EXCEL_PR_SHEETS.items():
        if sheet not in xl.sheet_names:
            logger.warning(f'  Sheet not found: {sheet}')
            continue
        try:
            parsed = _parse_pr_sheet(xl, sheet, resource)
            for (res, metric), scale in EXCEL_METRIC_SCALE.items():
                if res == resource:
                    mask = parsed['Metric'] == metric
                    if mask.any():
                        parsed.loc[mask, 'Value'] *= scale
            logger.info(f'  {sheet}: {len(parsed):,} rows')
            frames.append(parsed)
        except Exception as e:
            logger.error(f'  Failed to parse {sheet}: {e}')

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)

## Source 2b -- EI Excel price sheets

In [9]:
def load_ei_excel_prices(path: Path) -> pd.DataFrame:
    logger.info('Loading EI Excel price sheets...')
    if not path.exists():
        return pd.DataFrame(columns=FINAL_COLS)

    xl     = pd.ExcelFile(path)
    frames = []

    # Oil crude prices (world, $/bbl, back to 1861)
    if 'Oil crude prices since 1861' in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name='Oil crude prices since 1861', header=None)
        price_df = df.iloc[4:, [0, 2]].copy()
        price_df.columns = ['Year', 'Value']
        price_df = price_df.dropna()
        price_df['Country']  = 'World'
        price_df['Resource'] = 'Oil'
        price_df['Metric']   = 'Price'
        frames.append(price_df[FINAL_COLS])
        logger.info(f'  Oil prices: {len(price_df):,} rows')

    # Coal prices (NW Europe benchmark, $/tonne)
    if 'Coal & Uranium - Prices' in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name='Coal & Uranium - Prices', header=None)
        coal_df = df.iloc[4:, [0, 3]].copy()
        coal_df.columns = ['Year', 'Value']
        coal_df = coal_df[pd.to_numeric(coal_df['Year'], errors='coerce').notna()]
        coal_df['Value'] = pd.to_numeric(coal_df['Value'], errors='coerce')
        coal_df = coal_df.dropna(subset=['Value'])
        coal_df['Country']  = 'World'
        coal_df['Resource'] = 'Coal'
        coal_df['Metric']   = 'Price'
        frames.append(coal_df[FINAL_COLS])
        logger.info(f'  Coal prices: {len(coal_df):,} rows')

    # Mineral commodity prices (Cobalt, Lithium, Nickel, Natural Graphite)
    if 'Mineral Commodity Prices' in xl.sheet_names:
        df      = pd.read_excel(xl, sheet_name='Mineral Commodity Prices', header=None)
        headers = df.iloc[3].tolist()
        resource_col_map = {}
        for ci, h in enumerate(headers):
            h_str = str(h).lower()
            if 'cobalt'   in h_str: resource_col_map[ci] = 'Cobalt'
            elif 'lithium' in h_str: resource_col_map[ci] = 'Lithium'
            elif 'nickel'  in h_str: resource_col_map[ci] = 'Nickel'
            elif 'graphite' in h_str: resource_col_map[ci] = 'Natural Graphite'

        data     = df.iloc[4:].copy()
        year_col = data.iloc[:, 0]
        for ci, resource in resource_col_map.items():
            sub = pd.DataFrame({'Year': year_col, 'Value': data.iloc[:, ci]})
            sub['Year']  = pd.to_numeric(sub['Year'],  errors='coerce')
            sub['Value'] = pd.to_numeric(sub['Value'], errors='coerce')
            sub = sub.dropna()
            sub['Country']  = 'World'
            sub['Resource'] = resource
            sub['Metric']   = 'Price'
            frames.append(sub[FINAL_COLS])
        logger.info(f'  Mineral commodity prices: {sum(len(f) for f in frames):,} rows total')

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)

## Source 3 -- OWID mineral CSVs

In [10]:
def _infer_metric_from_column(col_name: str) -> str:
    col_lower = col_name.lower()
    for pattern, metric in OWID_METRIC_PATTERNS.items():
        if pattern in col_lower:
            return metric
    return 'Production'


def _infer_resource_from_column(col_name: str) -> str:
    parts = col_name.split('|')
    if len(parts) >= 2:
        return parts[1].strip().title()
    return col_name


def load_owid_csv(filepath: Path, resource: str, metric: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        logger.warning(f'  Could not read {filepath.name}: {e}')
        return pd.DataFrame(columns=FINAL_COLS)

    if df.empty:
        return pd.DataFrame(columns=FINAL_COLS)

    value_cols = [c for c in df.columns if c not in ('Entity', 'Code', 'Year')]
    if not value_cols:
        logger.warning(f'  No value column in {filepath.name}')
        return pd.DataFrame(columns=FINAL_COLS)

    out = df[['Entity', 'Year', value_cols[0]]].copy()
    out.columns = ['Country', 'Year', 'Value']
    out['Resource'] = resource
    out['Metric']   = metric
    return out[FINAL_COLS]


def load_all_owid(minerals_dir: Path) -> pd.DataFrame:
    logger.info('Loading OWID mineral CSVs...')
    if not minerals_dir.exists():
        logger.warning(f'Minerals directory not found: {minerals_dir}')
        return pd.DataFrame(columns=FINAL_COLS)

    subfolders = [p for p in sorted(minerals_dir.iterdir()) if p.is_dir()]
    if not subfolders:
        logger.warning(f'No subfolders found in {minerals_dir}')
        return pd.DataFrame(columns=FINAL_COLS)

    frames, skipped = [], []
    for folder in subfolders:
        folder_name = folder.name
        csv_path    = folder / f'{folder_name}.csv'
        if not csv_path.exists():
            candidates = [f for f in folder.glob('*.csv') if 'metadata' not in f.name.lower()]
            if not candidates:
                skipped.append(folder_name)
                continue
            csv_path = candidates[0]

        if folder_name in OWID_FILE_MAP:
            resource, metric = OWID_FILE_MAP[folder_name]
        else:
            try:
                df         = pd.read_csv(csv_path, nrows=1)
                value_cols = [c for c in df.columns if c not in ('Entity', 'Code', 'Year')]
                if not value_cols:
                    skipped.append(folder_name)
                    continue
                resource = _infer_resource_from_column(value_cols[0])
                metric   = _infer_metric_from_column(value_cols[0])
            except Exception:
                skipped.append(folder_name)
                continue

        parsed = load_owid_csv(csv_path, resource, metric)
        if not parsed.empty:
            frames.append(parsed)
            logger.info(f'  {folder_name}: {len(parsed):,} rows ({resource} | {metric})')
        else:
            skipped.append(folder_name)

    if skipped:
        logger.info(f'  Skipped (empty/unreadable): {skipped}')

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)

## Source 4 -- USGS ds140 mineral prices (NEW)

Reads `ds140-*.xlsx` files from `USGS_DIR`. Each file contains annual unit values in $/t
for a single mineral. These are the standard reference prices used by USGS and are generally
more complete than the EI mineral commodity price sheet (which only covers Cobalt, Lithium,
Nickel, Natural Graphite).

The USGS files have 4 header rows to skip, a `Year` column, and a `Unit value ($/t)` column.
The mineral name is inferred from the filename (e.g. `ds140-coppe.xlsx` -> Copper).

In [11]:
def _resolve_usgs_name(filename_stem: str) -> str:
    """Map ds140 filename stem to pipeline resource name."""
    # stem is e.g. 'ds140-coppe'
    parts = filename_stem.split('-')
    if len(parts) < 2:
        return filename_stem
    mineral_key = parts[1].lower()
    # Try exact match first, then prefix match
    if mineral_key in USGS_NAME_MAP:
        return USGS_NAME_MAP[mineral_key]
    for prefix, resource in USGS_NAME_MAP.items():
        if mineral_key.startswith(prefix):
            return resource
    return mineral_key.title()


def load_usgs_prices(usgs_dir: Path) -> pd.DataFrame:
    logger.info('Loading USGS ds140 mineral prices...')
    if not usgs_dir.exists():
        logger.warning(f'USGS directory not found: {usgs_dir}')
        return pd.DataFrame(columns=FINAL_COLS)

    files = sorted(usgs_dir.glob('ds140-*.xlsx'))
    if not files:
        logger.warning(f'No ds140-*.xlsx files found in {usgs_dir}')
        return pd.DataFrame(columns=FINAL_COLS)

    frames = []
    for file in files:
        resource = _resolve_usgs_name(file.stem)
        try:
            df = pd.read_excel(file, skiprows=4)
        except Exception as e:
            logger.warning(f'  Could not read {file.name}: {e}')
            continue

        # Remove trailing footnote rows by keeping only rows with a numeric first column
        year_col = df.columns[0]
        df = df[pd.to_numeric(df[year_col], errors='coerce').notna()].copy()

        # Find the unit value column
        uv_cols = [c for c in df.columns if 'unit value' in str(c).lower()]
        if not uv_cols:
            logger.warning(f'  No unit value column in {file.name}')
            continue

        sub = df[[year_col, uv_cols[0]]].copy()
        sub.columns = ['Year', 'Value']
        sub['Year']  = pd.to_numeric(sub['Year'],  errors='coerce')
        sub['Value'] = pd.to_numeric(sub['Value'], errors='coerce')
        sub = sub.dropna(subset=['Year', 'Value'])

        sub['Country']  = 'World'
        sub['Resource'] = resource
        sub['Metric']   = 'Price'
        frames.append(sub[FINAL_COLS])
        logger.info(f'  {file.name} -> {resource}: {len(sub):,} rows')

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    result = pd.concat(frames, ignore_index=True)
    logger.info(f'  USGS total: {len(result):,} price rows across {len(frames)} minerals')
    return result

## Source 5 -- Oil / Gas / Coal price benchmarks

Reads `rawdata/Oil Gas Coal Uranium Price.xlsx` and extracts three world-level price series:

| Series | Columns used | Unit | Coverage |
|---|---|---|---|
| Natural Gas | Henry Hub, TTF, JKM (averaged) | $/MMBtu | 1989–2024 |
| Oil | Brent | $/bbl | 1976–2024 |
| Coal | PCOALAUUSDA (Australian benchmark) | $/tonne | 1990–2024 |

All three are tagged `Country='World'` under source label `GasPrice`.
In the priority order they sit **below** `ConsolidatedPrices` and `EI_Prices`, so they only contribute year-gaps not already covered by those sources (e.g. Oil years before 1990 that fall outside the consolidated file's range).

In [12]:
def load_gas_prices(path: Path) -> pd.DataFrame:
    """
    Reads rawdata/Oil Gas Coal Uranium Price.xlsx.
    Extracts:
      - Natural Gas price  : average of Henry Hub / TTF / JKM ($/MMBtu)
      - Oil price          : Brent ($/bbl)        — fills gap years not in EI_Prices
      - Coal price         : PCOALAUUSDA ($/tonne) — fills gap years not in EI_Prices
    All three are tagged Country='World' and slotted into the GasPrice source,
    which sits below ConsolidatedPrices and EI_Prices in priority so they only
    fill genuine gaps rather than overriding better series.
    """
    logger.info('Loading gas/oil/coal price benchmarks from Oil Gas Coal Uranium Price.xlsx...')
    if not path.exists():
        logger.warning(f'Gas price file not found: {path}')
        return pd.DataFrame(columns=FINAL_COLS)

    try:
        df = pd.read_excel(path)
    except Exception as e:
        logger.warning(f'  Could not read gas price file: {e}')
        return pd.DataFrame(columns=FINAL_COLS)

    year_col = 'Year'
    if year_col not in df.columns:
        logger.warning('  No Year column found in gas price file')
        return pd.DataFrame(columns=FINAL_COLS)

    df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
    df = df.dropna(subset=[year_col])
    df[year_col] = df[year_col].astype(int)

    frames = []

    gas_cols = [c for c in df.columns if any(k in str(c).lower() for k in ('gas', 'henry', 'ttf', 'jkm'))]
    if gas_cols:
        logger.info(f'  Gas benchmark columns: {gas_cols}')
        df['_gas_avg'] = pd.to_numeric(df[gas_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1), errors='coerce')
        sub = df[[year_col, '_gas_avg']].rename(columns={year_col: 'Year', '_gas_avg': 'Value'}).copy()
        sub['Value'] = pd.to_numeric(sub['Value'], errors='coerce')
        sub = sub.dropna(subset=['Value'])
        sub['Country'] = 'World'; sub['Resource'] = 'Natural Gas'; sub['Metric'] = 'Price'
        frames.append(sub[FINAL_COLS])
        logger.info(f'  Natural Gas price: {len(sub):,} rows ({int(sub["Year"].min())}-{int(sub["Year"].max())})')
    else:
        logger.warning('  No gas benchmark columns found')

    brent_cols = [c for c in df.columns if 'brent' in str(c).lower() or 'barrel' in str(c).lower()]
    if brent_cols:
        sub = df[[year_col, brent_cols[0]]].rename(columns={year_col: 'Year', brent_cols[0]: 'Value'}).copy()
        sub['Value'] = pd.to_numeric(sub['Value'], errors='coerce')
        sub = sub.dropna(subset=['Value'])
        sub['Country'] = 'World'; sub['Resource'] = 'Oil'; sub['Metric'] = 'Price'
        frames.append(sub[FINAL_COLS])
        logger.info(f'  Oil (Brent) price: {len(sub):,} rows ({int(sub["Year"].min())}-{int(sub["Year"].max())})')

    coal_cols = [c for c in df.columns if 'coal' in str(c).lower()]
    if coal_cols:
        sub = df[[year_col, coal_cols[0]]].rename(columns={year_col: 'Year', coal_cols[0]: 'Value'}).copy()
        sub['Value'] = pd.to_numeric(sub['Value'], errors='coerce')
        sub = sub.dropna(subset=['Value'])
        sub['Country'] = 'World'; sub['Resource'] = 'Coal'; sub['Metric'] = 'Price'
        frames.append(sub[FINAL_COLS])
        logger.info(f'  Coal price: {len(sub):,} rows ({int(sub["Year"].min())}-{int(sub["Year"].max())})')

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)

    result = pd.concat(frames, ignore_index=True)
    logger.info(f'  GasPrice source total: {len(result):,} rows')
    return result

In [13]:
# Keys are row labels in workingdata/natural resource prices.xlsx (spaces, not underscores);
# values are pipeline resource names. Phosphate, Uranium, and individual gas
# benchmarks (Henry Hub, TTF, JKM) are excluded — the consolidated file already
# has a pre-averaged 'Gas' row.
CONSOL_RESOURCE_MAP = {
    'bauxite':                    'Bauxite',
    'cobalt':                     'Cobalt',
    'copper':                     'Copper',
    'gold':                       'Gold',
    'graphite':                   'Natural Graphite',
    'iron_ore':                   'Iron ore',
    'lead':                       'Lead',
    'lithium':                    'Lithium',
    'manganese':                  'Manganese',
    'nickel':                     'Nickel',
    'platinum':                   'Platinum Group',
    'rare':                       'Rare Earth',
    'silver':                     'Silver',
    'tin':                        'Tin',
    'vanadium':                   'Vanadium',
    'zinc':                       'Zinc',
    'Brent (Dollars per Barrel)': 'Oil',           # $/bbl — matches EI convention
    'Coal (PCOALAUUSDA)':         'Coal',           # $/tonne — Australian benchmark
    'Gas':                        'Natural Gas',    # $/MMBtu — pre-averaged HH/TTF/JKM
}
# Case-insensitive fallback so minor capitalisation differences in the Excel file
# (e.g. 'Bauxite' vs 'bauxite') don't silently drop rows.
_CONSOL_RESOURCE_MAP_LOWER = {k.lower(): v for k, v in CONSOL_RESOURCE_MAP.items()}


def load_consolidated_prices(path: Path) -> pd.DataFrame:
    """
    Load the wide-format consolidated price file (workingdata/natural resource prices.xlsx).

    Layout: row 0 = years (cols 1+), rows 1+ = one resource per row.
    All mineral prices are in $/tonne; Oil in $/bbl; Gas in $/MMBtu.
    Rows not in CONSOL_RESOURCE_MAP (Phosphate, Uranium, individual gas benchmarks) are skipped.
    """
    logger.info('Loading consolidated price file...')
    if not path.exists():
        logger.warning(f'Consolidated price file not found: {path}')
        return pd.DataFrame(columns=FINAL_COLS)

    try:
        df = pd.read_excel(path, header=None)
    except Exception as e:
        logger.warning(f'  Could not read consolidated price file: {e}')
        return pd.DataFrame(columns=FINAL_COLS)

    # Row 0 = year headers; col 0 = resource label
    year_row = df.iloc[0, 1:]
    years = pd.to_numeric(year_row, errors='coerce')

    frames = []
    for i in range(1, len(df)):
        label = str(df.iloc[i, 0]).strip()
        # Exact match first; case-insensitive fallback for minor capitalisation differences
        resource = CONSOL_RESOURCE_MAP.get(label) or _CONSOL_RESOURCE_MAP_LOWER.get(label.lower())
        if resource is None:
            continue  # skip phosphate, uranium, individual gas benchmarks

        values = pd.to_numeric(df.iloc[i, 1:], errors='coerce')
        sub = pd.DataFrame({'Year': years.values, 'Value': values.values})
        sub = sub.dropna(subset=['Year', 'Value'])
        sub['Year']     = sub['Year'].astype(int)
        sub['Country']  = 'World'
        sub['Resource'] = resource
        sub['Metric']   = 'Price'
        frames.append(sub[FINAL_COLS])
        logger.info(f'  {label} -> {resource}: {len(sub):,} rows '
                    f'({int(sub["Year"].min())}-{int(sub["Year"].max())})')

    if not frames:
        logger.warning('  No rows extracted from consolidated price file.')
        return pd.DataFrame(columns=FINAL_COLS)

    result = pd.concat(frames, ignore_index=True)
    logger.info(f'  ConsolidatedPrices total: {len(result):,} rows, '
                f'{result["Resource"].nunique()} resources')
    return result

## Combine and clean

In [14]:
def _canonical_name(name: str) -> str:
    """
    Map any country alias to its WB canonical name.
    Returns the original string unchanged if no ISO3 mapping exists
    (preserves 'World', 'Global', OPEC aggregates, etc.).
    """
    iso = ALIAS_TO_ISO3.get(str(name)) or ALIAS_TO_ISO3.get(str(name).strip())
    return ISO3_TO_WB.get(iso, name) if iso else name


def combine_and_clean(source_list, source_labels=None, conflict_path=None):
    logger.info('Combining and cleaning all sources...')

    if source_labels is None:
        source_labels = [f'Source_{i}' for i in range(len(source_list))]

    tagged = []
    for df, label in zip(source_list, source_labels):
        df = df.copy()
        df['Source'] = label
        tagged.append(df)

    combined = pd.concat(tagged, ignore_index=True)
    combined['Value'] = pd.to_numeric(combined['Value'], errors='coerce')
    combined['Year']  = pd.to_numeric(combined['Year'],  errors='coerce')
    combined = combined.dropna(subset=['Value', 'Year', 'Country', 'Resource', 'Metric'])
    combined['Year'] = combined['Year'].astype(int)
    for col in ['Country', 'Resource', 'Metric']:
        combined[col] = combined[col].str.strip()

    # Remove aggregates but keep World-level price rows
    agg_patterns = re.compile(r'(^total|^rest of|opec|oecd|european union|^eu$)', re.IGNORECASE)
    is_agg        = combined['Country'].str.match(agg_patterns, na=False)
    is_world_price = (combined['Country'] == 'World') & (combined['Metric'] == 'Price')
    combined      = combined[~is_agg | is_world_price]

    # Unit normalisation: OWID volumes -> EI canonical units (kbd / Mt / kt)
    mask_owid = combined['Source'] == 'OWID'
    for resource, factor in OWID_TO_CANONICAL.items():
        skip_reserves = resource in OWID_RESERVES_ALREADY_KT
        if skip_reserves:
            mask = mask_owid & (combined['Resource'] == resource) & (combined['Metric'] == 'Production')
        else:
            mask = mask_owid & (combined['Resource'] == resource) & (combined['Metric'] != 'Price')
        combined.loc[mask, 'Value'] *= factor
    logger.info('OWID values converted to EI canonical units.')

    # EI mineral prices are in $/kg; convert to $/tonne for consistency with OWID and USGS.
    MINERAL_PRICE_RESOURCES = {
        'Cobalt', 'Lithium', 'Nickel', 'Natural Graphite', 'Copper', 'Zinc', 'Aluminium',
        'Bauxite', 'Lead', 'Tin', 'Manganese', 'Silver', 'Gold', 'Cadmium', 'Vanadium',
        'Magnesium compounds', 'Iron ore', 'Rare Earth', 'Platinum Group',
    }
    mask_ei_price = (
        (combined['Source'] == 'EI_Prices') &
        (combined['Metric'] == 'Price') &
        (combined['Resource'].isin(MINERAL_PRICE_RESOURCES))
    )
    combined.loc[mask_ei_price, 'Value'] *= 1_000
    logger.info(f'EI_Prices mineral prices converted $/kg -> $/tonne ({mask_ei_price.sum()} rows).')

    # Canonicalize country names before deduplication
    ref_rows = combined['Country'].isin(['World', 'Global'])
    combined.loc[~ref_rows, 'Country'] = combined.loc[~ref_rows, 'Country'].map(_canonical_name)
    logger.info('Country names canonicalized to WB standard before deduplication.')

    combined['_priority'] = combined.apply(
        lambda r: _priority(r['Resource'], r['Metric'], r['Source']), axis=1
    )

    # For production/consumption/reserves rows, prefer the source that has the
    # most non-null years for that specific (country, resource, metric) group.
    # This lets OWID win for a country where it has 30 years vs EI's 10, even if
    # EI is globally ranked higher. Price rows are excluded — they are world-level
    # and source quality (priority) matters more than year-count there.
    is_price = combined['Metric'] == 'Price'

    coverage = (
        combined[~is_price & combined['Value'].notna()]
        .groupby(['Country', 'Resource', 'Metric', 'Source'])
        .size()
        .reset_index(name='_n_years')
    )
    combined = combined.merge(coverage, on=['Country', 'Resource', 'Metric', 'Source'], how='left')
    combined['_n_years'] = combined['_n_years'].fillna(0).astype(int)
    # Price rows get _n_years=0, so they sort entirely by _priority — correct behaviour.

    # Sort: higher coverage first, then lower (better) priority number
    combined = combined.sort_values(
        ['Resource', 'Country', 'Year', 'Metric', '_n_years', '_priority'],
        ascending=[True, True, True, True, False, True]
    ).reset_index(drop=True)

    # Conflict report
    key_cols  = ['Country', 'Year', 'Resource', 'Metric']
    dupe_mask = combined.duplicated(subset=key_cols, keep=False)
    dupes     = combined[dupe_mask].copy()

    if len(dupes) > 0:
        conflicts = []
        for key, group in dupes.groupby(key_cols):
            primary = group.iloc[0]
            for _, alt in group.iloc[1:].iterrows():
                pct = (
                    0.0 if primary['Value'] == 0 and alt['Value'] == 0
                    else float('inf') if primary['Value'] == 0
                    else abs(primary['Value'] - alt['Value']) / abs(primary['Value']) * 100
                )
                conflicts.append({
                    'Country':       key[0], 'Year': key[1],
                    'Resource':      key[2], 'Metric': key[3],
                    'Kept_Source':   primary['Source'],
                    'Kept_n_years':  int(primary['_n_years']),
                    'Kept_Value':    primary['Value'],
                    'Dropped_Source':  alt['Source'],
                    'Dropped_n_years': int(alt['_n_years']),
                    'Dropped_Value':   alt['Value'],
                    'Pct_Discrepancy': round(pct, 1),
                })
        conflict_df    = pd.DataFrame(conflicts)
        real_conflicts = conflict_df[conflict_df['Pct_Discrepancy'] > 0]
        # Flag cases where coverage tiebreak overrode global priority
        coverage_overrides = conflict_df[
            (conflict_df['Kept_n_years'] > conflict_df['Dropped_n_years'])
        ]
        logger.info(
            f'Conflicts: {len(real_conflicts):,} real (>0%), '
            f'{(real_conflicts["Pct_Discrepancy"] > 10).sum()} >10%, '
            f'{(real_conflicts["Pct_Discrepancy"] > 50).sum()} >50%'
        )
        logger.info(
            f'Coverage-based overrides (lower-priority source kept due to better '
            f'per-country coverage): {len(coverage_overrides):,}'
        )
        out_path = conflict_path if conflict_path is not None else OUTPUT_PATH.parent / 'NR_source_conflicts.csv'
        conflict_df.to_csv(out_path, index=False)
        logger.info(f'Conflict log saved to {out_path}')

    before   = len(combined)
    combined = combined.drop_duplicates(subset=key_cols, keep='first')
    combined = combined.drop(columns=['_priority', '_n_years', 'Source'])
    logger.info(f'Deduplication: {before:,} -> {len(combined):,} rows ({before - len(combined):,} removed)')

    return combined.sort_values(['Resource', 'Country', 'Year']).reset_index(drop=True)

## Production value computation

For each (Country, Resource, Year) where both a Production row and a Price row exist,
compute:

    ProductionValue_USD = Production * Price * unit_multiplier

Prices are world-level (Country='World'), so they are broadcast across all countries.
The unit multiplier converts the product of (production_unit * price_unit) into USD:

| Resource | Production unit | Price unit | Multiplier | Result |
|---|---|---|---|---|
| Oil | kbd | $/bbl | 365,000 | $/yr |
| Natural Gas | bcm | $/MMBtu | ~36.6M | $/yr |
| Coal | Mt | $/tonne | 1,000,000 | $/yr |
| Minerals | kt | $/tonne | 1,000 | $/yr |

In [15]:
def compute_production_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given the deduplicated long-format panel, compute production values in USD.
    Returns a new DataFrame with Metric='ProductionValue' rows appended.
    """
    logger.info('Computing production values...')

    prod   = df[df['Metric'] == 'Production'].copy()
    prices = df[(df['Metric'] == 'Price') & (df['Country'] == 'World')].copy()

    if prices.empty:
        logger.warning('No world-level price rows found. Cannot compute production values.')
        return df

    # Vectorized price merge — replaces the previous row-wise .apply() price lookup
    world_prices = prices[['Resource', 'Year', 'Value']].rename(columns={'Value': 'Price'})
    logger.info(f'  Price lookup: {len(world_prices):,} (Resource, Year) pairs')

    prod = prod.merge(world_prices, on=['Resource', 'Year'], how='left')

    # Drop rows without matching prices
    has_price = prod['Price'].notna()
    logger.info(f'  Production rows with matching price: {has_price.sum():,} / {len(prod):,}')
    prod = prod[has_price].copy()

    # Apply unit multipliers
    prod['Multiplier'] = prod['Resource'].map(
        lambda r: PRODVAL_MULTIPLIER.get(r, MINERAL_PRODVAL_MULT)
    )
    prod['Value'] = prod['Value'] * prod['Price'] * prod['Multiplier']
    prod['Metric'] = 'ProductionValue'
    prod = prod[FINAL_COLS]

    logger.info(f'  Computed {len(prod):,} production value rows')
    logger.info(f'  Resources covered: {sorted(prod["Resource"].unique())}')

    # Coverage report: which resources have prices vs not
    all_prod_resources = set(df[df['Metric'] == 'Production']['Resource'].unique())
    priced_resources   = set(prices['Resource'].unique())
    unpriced = all_prod_resources - priced_resources
    if unpriced:
        logger.warning(f'  Resources without any price data (no production value): {sorted(unpriced)}')

    return pd.concat([df, prod], ignore_index=True).sort_values(
        ['Resource', 'Country', 'Year', 'Metric']
    ).reset_index(drop=True)

## Validation

In [16]:
def validate(df: pd.DataFrame) -> None:
    logger.info('\n--- Validation Report ---')
    logger.info(f'Total rows:  {len(df):,}')
    logger.info(f'Countries:   {df["Country"].nunique()}')
    logger.info(f'Resources:   {sorted(df["Resource"].unique())}')
    logger.info(f'Metrics:     {sorted(df["Metric"].unique())}')
    logger.info(f'Year range:  {df["Year"].min()} -- {df["Year"].max()}')

    russia = df[df['Country'].isin(['Russia', 'Russian Federation'])]
    logger.info(f'Russia rows: {len(russia):,} (as Russian Federation: {len(russia[russia["Country"]=="Russian Federation"]):,})')

    expected = {(r, m) for r, m in EI_CSV_VAR_MAP.values()}
    found    = set(zip(df['Resource'], df['Metric']))
    missing  = expected - found
    if missing:
        logger.warning(f'Expected resource-metric pairs missing from output: {missing}')

    al_res = len(df[(df['Resource'] == 'Aluminium') & (df['Metric'] == 'Reserves')])
    if al_res == 0:
        logger.warning('Aluminium has zero Reserves rows -- check Capacity column detection')
    else:
        logger.info(f'Aluminium Reserves rows: {al_res} (Capacity column parsed correctly)')

    # Production value coverage
    pv = df[df['Metric'] == 'ProductionValue']
    if not pv.empty:
        logger.info(f'ProductionValue rows: {len(pv):,}')
        logger.info(f'  Countries with PV: {pv["Country"].nunique()}')
        logger.info(f'  Resources with PV: {sorted(pv["Resource"].unique())}')
        total_usd = pv['Value'].sum()
        logger.info(f'  Total production value (all countries, all years): ${total_usd:,.0f}')
    else:
        logger.warning('No ProductionValue rows found.')

    logger.info('-------------------------\n')

## Run

In [17]:
logger.info('=' * 55)
logger.info('BUILD: natural_resources_production_values.csv')
logger.info('=' * 55)

ei_csv      = load_ei_csv(EI_CSV_PATH)
ei_minerals = load_ei_excel_minerals(EI_XLSX_PATH)
ei_prices   = load_ei_excel_prices(EI_XLSX_PATH)
owid        = load_all_owid(MINERALS_DIR)
usgs        = load_usgs_prices(USGS_DIR)        # graceful no-op if USGS dir absent
gas         = load_gas_prices(GAS_PRICE_PATH)   # graceful no-op if file absent
consol      = load_consolidated_prices(CONSOL_PRICE_PATH)  # consolidated price file

final = combine_and_clean(
    [ei_csv, ei_minerals, ei_prices, owid, usgs, gas, consol],
    source_labels=['EI_CSV', 'EI_Excel', 'EI_Prices', 'OWID', 'USGS', 'GasPrice', 'ConsolidatedPrices'],
)

final = compute_production_values(final)

validate(final)

final = add_iso3(final, name_col='Country', iso3_col='iso3')
unmatched = final[final['iso3'].isna() & ~final['Country'].isin(['World', 'Global'])]['Country'].unique()
if len(unmatched) > 0:
    logger.warning(f'Unmatched country names ({len(unmatched)}): {sorted(unmatched)[:15]}')
logger.info(f'ISO3 coverage: {final["iso3"].notna().sum():,} / {len(final):,} rows')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Full panel (all metrics including ProductionValue)
final.to_csv(PRODVAL_PATH, index=False)
logger.info(f'Saved {len(final):,} rows to {PRODVAL_PATH}')

# Backward-compatible output (without ProductionValue rows)
compat = final[final['Metric'] != 'ProductionValue']
compat.to_csv(OUTPUT_PATH, index=False)
logger.info(f'Saved {len(compat):,} rows to {OUTPUT_PATH} (backward-compatible)')

INFO: =======================================================
INFO: BUILD: natural_resources_production_values.csv
INFO: =======================================================
INFO: Loading EI Narrow CSV...
INFO:   EI CSV: 28,473 rows across 14 variables
INFO: Loading EI Excel mineral P-R sheets...
INFO:   Cobalt P-R: 403 rows
INFO:   Lithium P-R: 279 rows
INFO:   Natural Graphite P-R: 372 rows
INFO:   Rare Earth metals P-R: 279 rows
INFO:   Copper P-R: 96 rows
INFO:   Manganese P-R: 84 rows
INFO:   Nickel P-R: 104 rows
INFO:   Zinc P-R: 91 rows
INFO:   Platinum Group Metals P-R: 84 rows
INFO:   Bauxite P-R: 108 rows
INFO:   Aluminium P-R: 84 rows
INFO:   Tin P-R: 108 rows
INFO:   Vanadium P-R: 84 rows
INFO: Loading EI Excel price sheets...
INFO:   Oil prices: 164 rows
INFO:   Coal prices: 38 rows
INFO:   Mineral commodity prices: 282 rows total
INFO: Loading OWID mineral CSVs...
INFO:   aluminum-production: 2,420 rows (Aluminium | Production)
INFO:   aluminum-unit-value: 122 rows (Al